In [7]:
!pip install mlflow

import os
import shutil
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

def train_heart_model(data_path="/content/sample_data/heart.csv"):
    mlflow.set_experiment("heart-disease-prediction")

    # Load Heart Disease CSV dataset
    df = pd.read_csv(data_path)
    X = df.drop(columns=["target"])
    y = df["target"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    params = {
        "n_estimators": 150,
        "max_depth": 6,
        "min_samples_split": 4,
        "random_state": 42,
    }

    with mlflow.start_run() as run:
        # Preprocessing + Classifier Pipeline
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", RandomForestClassifier(**params))
        ])

        pipeline.fit(X_train, y_train)

        # Predictions & Evaluation
        preds = pipeline.predict(X_test)
        probs = pipeline.predict_proba(X_test)[:, 1]
        acc = accuracy_score(y_test, preds)
        auc = roc_auc_score(y_test, probs)

        # Log to MLflow
        mlflow.log_params(params)
        mlflow.log_param("dataset_rows", len(df))
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)

        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path="model",
            registered_model_name="HeartDiseaseClassifier",
            skops_trusted_types=["sklearn.tree._tree.Tree"]
        )

        # Save standalone model for FastAPI / Docker
        save_path = "artifacts/heart_model"
        if os.path.exists(save_path):
            shutil.rmtree(save_path)

        os.makedirs("artifacts", exist_ok=True)
        mlflow.sklearn.save_model(
            sk_model=pipeline,
            path=save_path,
            mlflow_model=None,
            skops_trusted_types=["sklearn.tree._tree.Tree"]
        )

        print(f"Run ID: {run.info.run_id}")
        print(f"Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")
        print(classification_report(y_test, preds))

if __name__ == "__main__":
    train_heart_model()

2026/11/14 15:33:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'HeartDiseaseClassifier' already exists. Creating a new version of this model...
Created version '3' of model 'HeartDiseaseClassifier'.


Run ID: 4579a086ddbe472ea092afd260d12c52
Accuracy: 0.9659 | ROC-AUC: 0.9867
              precision    recall  f1-score   support

           0       0.97      0.96      0.96       100
           1       0.96      0.97      0.97       105

    accuracy                           0.97       205
   macro avg       0.97      0.97      0.97       205
weighted avg       0.97      0.97      0.97       205

